# Enterprise Industrial AI Copilot

## Notebook 01

# Enterprise Document Discovery & Inventory Pipeline

---

### Author

Charan Teja Arangi

---

### Objective

Build a production-grade enterprise document discovery pipeline capable of scanning an industrial repository, validating documents, generating metadata, creating inventory datasets, and preparing the foundation for OCR, RAG, Knowledge Graph, and Semantic Search pipelines.

---

### Input

data/raw/

---

### Outputs

- document_inventory.csv
- file_hashes.csv
- dataset_statistics.json
- folder_summary.json
- processing_log.txt

In [1]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
import hashlib
import mimetypes
import logging
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================================
# Universal Notebook Setup
# Compatible with VS Code • GitHub • Local Development
# ==========================================================

import sys
from pathlib import Path

# ----------------------------------------------------------
# Step 1: Automatically locate PROJECT_ROOT
# ----------------------------------------------------------
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = None

# Search upwards (max 5 levels) for src/core/config.py
for _ in range(5):
    if (_current_dir / "src" / "core" / "config.py").is_file():
        PROJECT_ROOT = _current_dir
        break
    _current_dir = _current_dir.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "❌ Could not automatically determine PROJECT_ROOT.\n"\
        "Please make sure the notebook is opened from inside the project."
    )

# ----------------------------------------------------------
# Step 2: Add PROJECT_ROOT to Python path (if needed)
# ----------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ----------------------------------------------------------
# Step 3: Import centralized configuration
# ----------------------------------------------------------
from src.core.config import (
    PROJECT_ROOT as CONFIG_ROOT,
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,
)

# ----------------------------------------------------------
# Step 4: Verify PROJECT_ROOT consistency
# ----------------------------------------------------------
if CONFIG_ROOT != PROJECT_ROOT:
    print("⚠ WARNING: Notebook PROJECT_ROOT differs from config.py PROJECT_ROOT")
    print(f"Notebook : {PROJECT_ROOT}")
    print(f"Config   : {CONFIG_ROOT}")

# Use the centralized PROJECT_ROOT from config.py
PROJECT_ROOT = CONFIG_ROOT

# ----------------------------------------------------------
# Step 5: Verify required directories exist
# ----------------------------------------------------------
required_dirs = [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,
]

missing_dirs = [d for d in required_dirs if not d.exists()]

if missing_dirs:
    print("Missing Directories:")
    for d in missing_dirs:
        print(f"   - {d}")
    raise FileNotFoundError("One or more required directories are missing.")

# ----------------------------------------------------------
# Step 6: Environment Verification
# ----------------------------------------------------------
print("=" * 65)
print("PROJECT SETUP VERIFICATION SUMMARY")
print("=" * 65)
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"DATA_DIR          : {DATA_DIR}")
print(f"RAW_DIR           : {RAW_DIR}")
print(f"PROCESSED_DIR     : {PROCESSED_DIR}")
print(f"CHUNK_DIR         : {CHUNK_DIR}")
print(f"EMBEDDING_DIR     : {EMBEDDING_DIR}")
print(f"VECTOR_DB_DIR     : {VECTOR_DB_DIR}")
print(f"KG_DIR            : {KG_DIR}")
print(f"INVENTORY_DIR     : {INVENTORY_DIR}")
print(f"LOG_DIR           : {LOG_DIR}")
print("-" * 65)
print("✅ Status          : SUCCESS")
print("=" * 65)


PROJECT SETUP VERIFICATION SUMMARY
PROJECT_ROOT      : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro
DATA_DIR          : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data
RAW_DIR           : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\raw
PROCESSED_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\processed
CHUNK_DIR         : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\chunks
EMBEDDING_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\embeddings
VECTOR_DB_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\vector_db
KG_DIR            : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\knowledge_graph
INVENTORY_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory
LOG_DIR           : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\logs
-----------------------------------------------------------------
✅ Status          : SUCCESS


In [3]:
# ==========================================================
# Create Output Directories
# ==========================================================

INVENTORY_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Directories Ready")

Directories Ready


In [4]:
# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(

    filename=LOG_DIR/"processing_log.txt",

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    force=True

)

logging.info("="*80)
logging.info("Notebook 01 Started")
logging.info("="*80)

print("Logger Ready")

Logger Ready


In [5]:
# ==========================================================
# Validate Folder Structure
# ==========================================================

required = [

    PROJECT_ROOT,

    DATA_DIR,

    RAW_DIR

]

missing = []

for folder in required:

    if not folder.exists():

        missing.append(str(folder))

if len(missing):

    raise FileNotFoundError(

        f"Missing folders:\n{missing}"

    )

print("Project Structure Verified")

Project Structure Verified


In [6]:
# ==========================================================
# Utility Function
# ==========================================================

def create_document_id(index):

    return f"DOC{index:05d}"

In [7]:
# ==========================================================
# Utility Function
# ==========================================================

def get_relative_path(path):

    return str(path.relative_to(PROJECT_ROOT))

In [8]:
# ==========================================================
# Utility Function
# ==========================================================

def bytes_to_mb(size):

    return round(size/(1024*1024),2)

In [9]:
from src.core.config import SUPPORTED_EXTENSIONS
# ==========================================================
# Notebook Configuration Summary
# ==========================================================

print("="*60)

print("Enterprise Industrial AI Copilot")

print("="*60)

print(f"Project Root : {PROJECT_ROOT}")

print(f"Raw Folder   : {RAW_DIR}")

print(f"Output Folder: {INVENTORY_DIR}")

print(f"Extensions   : {SUPPORTED_EXTENSIONS}")

print("="*60)

Enterprise Industrial AI Copilot
Project Root : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro
Raw Folder   : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\raw
Output Folder: C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory
Extensions   : {'.pdf', '.txt', '.jpg', '.png', '.csv', '.jpeg'}


In [10]:
# ==========================================================
# Stage Tracker
# ==========================================================

PIPELINE_STAGES = [

"Document Discovery",

"Validation",

"Metadata Extraction",

"SHA256",

"Duplicate Detection",

"Inventory",

"Statistics",

"Export"

]

for stage in PIPELINE_STAGES:

    print("✓",stage)

✓ Document Discovery
✓ Validation
✓ Metadata Extraction
✓ SHA256
✓ Duplicate Detection
✓ Inventory
✓ Statistics
✓ Export


In [11]:
# ==========================================================
# Initial Checks
# ==========================================================

assert RAW_DIR.exists()

assert INVENTORY_DIR.exists()

assert LOG_DIR.exists()

print("Environment Ready")

Environment Ready


In [12]:
# ==========================================================
# Count Raw Files
# ==========================================================

all_files = [

    f

    for f in RAW_DIR.rglob("*")

    if f.is_file()

]

print("Total Raw Files :",len(all_files))

Total Raw Files : 60


In [13]:
# ==========================================================
# Preview Raw Files
# ==========================================================

preview = []

for file in all_files[:20]:

    preview.append({

        "File":file.name,

        "Folder":file.parent.name

    })

pd.DataFrame(preview)

,File,Folder
0,.gitkeep,raw
1,990-5712_InRow RD DX Direct Expansion Air Cond...,maintenance
2,GEX2563900EN_PIXStandard.pdf,maintenance
3,JYT1925500 EVlink Preventive maintenance guide...,maintenance
4,MFR5408600 v03.00_Web.pdf,maintenance
5,MNT-MAN-HVAC-001.pdf,maintenance
6,OnSite Preventive Maintenance_SOW.pdf,maintenance
7,BBFACT01.pdf,safety_and_regulations
8,EHS-SOP-CONFINED-002.pdf,safety_and_regulations
9,EHS-SOP-LOTO-001.pdf,safety_and_regulations


In [14]:
print()

print("="*60)

print("MILESTONE 1 COMPLETED")

print("="*60)

logging.info("Milestone 1 Completed")


MILESTONE 1 COMPLETED


# Stage 1 — Enterprise Document Discovery

## Objective

This stage recursively scans the enterprise repository to discover all supported documents.

The discovery engine:

- Searches every subfolder recursively.
- Ignores hidden/system files.
- Filters unsupported file types.
- Creates deterministic Document IDs.
- Produces a clean inventory for downstream processing.

In [15]:
# ==========================================================
# Recursive Document Discovery
# ==========================================================

def discover_documents(root_dir, supported_extensions):
    """
    Recursively discover supported files.
    """

    discovered = []

    for file in root_dir.rglob("*"):

        if not file.is_file():
            continue

        if file.name.startswith("."):
            continue

        if file.suffix.lower() not in supported_extensions:
            continue

        discovered.append(file)

    discovered = sorted(discovered)

    logging.info(f"Discovered {len(discovered)} supported documents.")

    return discovered

In [16]:
from src.core.config import SUPPORTED_EXTENSIONS
# ==========================================================
# Execute Discovery
# ==========================================================

documents = discover_documents(
    RAW_DIR,
    SUPPORTED_EXTENSIONS
)

print("Supported Documents Found :", len(documents))

Supported Documents Found : 59


In [17]:
# ==========================================================
# Build Initial Inventory
# ==========================================================

inventory = []

for idx, file in enumerate(documents, start=1):

    inventory.append({

        "Document_ID": create_document_id(idx),

        "File_Name": file.name,

        "Extension": file.suffix.lower(),

        "Folder": file.parent.name,

        "Relative_Path": get_relative_path(file),

        "Absolute_Path": str(file)

    })

inventory_df = pd.DataFrame(inventory)

inventory_df.head()

,Document_ID,File_Name,Extension,Folder,Relative_Path,Absolute_Path
0,DOC00001,86261.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86261.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
1,DOC00002,86797.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86797.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
2,DOC00003,86798.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86798.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
3,DOC00004,86801.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86801.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
4,DOC00005,86808.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86808.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...


In [18]:
# ==========================================================
# Preview Inventory
# ==========================================================

print("="*70)
print("DOCUMENT INVENTORY PREVIEW")
print("="*70)

display(inventory_df.head(20))

DOCUMENT INVENTORY PREVIEW


,Document_ID,File_Name,Extension,Folder,Relative_Path,Absolute_Path
0,DOC00001,86261.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86261.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
1,DOC00002,86797.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86797.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
2,DOC00003,86798.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86798.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
3,DOC00004,86801.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86801.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
4,DOC00005,86808.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86808.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
5,DOC00006,86839.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86839.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
6,DOC00007,86860.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86860.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
7,DOC00008,86861.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86861.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
8,DOC00009,86868.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86868.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...
9,DOC00010,86877.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86877.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...


In [19]:
# ==========================================================
# Discovery Validation
# ==========================================================

assert len(inventory_df) == len(documents)

assert inventory_df["Document_ID"].is_unique

assert inventory_df["File_Name"].notna().all()

assert inventory_df["Relative_Path"].notna().all()

print("Discovery Validation Passed")

Discovery Validation Passed


In [20]:
# ==========================================================
# Folder Distribution
# ==========================================================

folder_summary = (

    inventory_df

    .groupby("Folder")

    .size()

    .reset_index(name="Count")

    .sort_values("Count", ascending=False)

)

display(folder_summary)

,Folder,Count
2,equipment_labels,12
6,safety_and_regulations,11
0,abb,7
4,inspection_forms,6
3,gauges,6
5,maintenance,6
7,schneider,5
1,atlas_copco,4
8,siemens,2


In [21]:
# ==========================================================
# Extension Distribution
# ==========================================================

extension_summary = (

    inventory_df

    .groupby("Extension")

    .size()

    .reset_index(name="Count")

    .sort_values("Count", ascending=False)

)

display(extension_summary)

,Extension,Count
1,.pdf,35
0,.jpg,21
2,.png,3


In [22]:
# ==========================================================
# Quick Statistics
# ==========================================================

print("="*60)

print("Discovery Statistics")

print("="*60)

print(f"Total Documents : {len(inventory_df)}")

print(f"Folders         : {inventory_df['Folder'].nunique()}")

print(f"Extensions      : {inventory_df['Extension'].nunique()}")

print("="*60)

Discovery Statistics
Total Documents : 59
Folders         : 9
Extensions      : 3


In [23]:
# ==========================================================
# Check Duplicate Paths
# ==========================================================

duplicate_paths = inventory_df.duplicated(
    subset=["Relative_Path"]
).sum()

print("Duplicate Paths :", duplicate_paths)

Duplicate Paths : 0


In [24]:
# ==========================================================
# Save Temporary Inventory
# ==========================================================

temp_inventory_path = INVENTORY_DIR / "inventory_stage1.csv"

inventory_df.to_csv(
    temp_inventory_path,
    index=False
)

print("Saved :", temp_inventory_path)

Saved : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory\inventory_stage1.csv


In [25]:
# ==========================================================
# Stage Completion
# ==========================================================

logging.info("Stage 1 Completed")

print()

print("="*60)

print("MILESTONE 2 COMPLETED")

print("="*60)


MILESTONE 2 COMPLETED


# Stage 2 — Enterprise Metadata Extraction

## Objective

This stage enriches every discovered enterprise document with technical metadata.

The generated metadata becomes the foundation for:

- Enterprise Search
- OCR Routing
- Document Analytics
- Knowledge Graph
- Retrieval Augmented Generation
- Compliance Intelligence

Output

Enterprise Metadata Table

In [26]:
# ==========================================================
# Metadata Extraction
# ==========================================================

def extract_metadata(file_path: Path):

    """
    Extract metadata from a document.

    Returns
    -------
    dict
    """

    stat = file_path.stat()

    mime_type, _ = mimetypes.guess_type(file_path)

    metadata = {

        "File_Size_Bytes": stat.st_size,

        "File_Size_MB": round(stat.st_size / (1024 * 1024), 2),

        "Last_Modified":

            datetime.fromtimestamp(

                stat.st_mtime

            ).strftime("%Y-%m-%d %H:%M:%S"),

        "MIME_Type":

            mime_type if mime_type else "Unknown",

        "Readable":

            os.access(file_path, os.R_OK)

    }

    return metadata

In [27]:
# ==========================================================
# Metadata Collection
# ==========================================================

metadata_records = []

for file in documents:

    metadata = extract_metadata(file)

    metadata_records.append(metadata)

metadata_df = pd.DataFrame(metadata_records)

metadata_df.head()

,File_Size_Bytes,File_Size_MB,Last_Modified,MIME_Type,Readable
0,10120,0.01,2026-07-11 15:33:19,image/jpeg,True
1,7263,0.01,2026-07-11 15:32:46,image/jpeg,True
2,7689,0.01,2026-07-11 15:32:59,image/jpeg,True
3,5798,0.01,2026-07-11 15:33:38,image/jpeg,True
4,6020,0.01,2026-07-11 15:33:31,image/jpeg,True


In [28]:
# ==========================================================
# Merge Metadata
# ==========================================================

inventory_df = pd.concat(

    [

        inventory_df,

        metadata_df

    ],

    axis=1

)

inventory_df.head()

,Document_ID,File_Name,Extension,Folder,Relative_Path,Absolute_Path,File_Size_Bytes,File_Size_MB,Last_Modified,MIME_Type,Readable
0,DOC00001,86261.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86261.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,10120,0.01,2026-07-11 15:33:19,image/jpeg,True
1,DOC00002,86797.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86797.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,7263,0.01,2026-07-11 15:32:46,image/jpeg,True
2,DOC00003,86798.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86798.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,7689,0.01,2026-07-11 15:32:59,image/jpeg,True
3,DOC00004,86801.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86801.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,5798,0.01,2026-07-11 15:33:38,image/jpeg,True
4,DOC00005,86808.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86808.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6020,0.01,2026-07-11 15:33:31,image/jpeg,True


In [29]:
# ==========================================================
# Enterprise Department Mapping
# ==========================================================

department_mapping = {

    # Engineering Manuals
    "abb": "Engineering",
    "siemens": "Engineering",
    "schneider": "Engineering",
    "atlas_copco": "Engineering",

    # Maintenance
    "maintenance": "Maintenance",

    # Safety
    "safety_and_regulations": "Safety",

    # Operations / OCR
    "equipment_labels": "Operations",
    "gauges": "Operations",
    "inspection_forms": "Quality"

}

inventory_df["Department"] = (

    inventory_df["Folder"]

    .map(department_mapping)

    .fillna("Unknown")

)

In [30]:
# ==========================================================
# Manufacturer Extraction
# ==========================================================

manufacturer_mapping = {

    "abb": "ABB",

    "siemens": "Siemens",

    "schneider": "Schneider Electric",

    "atlas_copco": "Atlas Copco"

}

inventory_df["Manufacturer"] = (

    inventory_df["Folder"]

    .map(manufacturer_mapping)

    .fillna("Nexus Industrial")

)

In [31]:
# ==========================================================
# Category Mapping
# ==========================================================

def classify_document(row):

    folder = row["Folder"]

    extension = row["Extension"]

    if extension==".pdf":

        if folder=="manuals":

            return "OEM Manual"

        elif folder=="maintenance":

            return "Maintenance Document"

        elif folder=="safety_and_regulations":

            return "Safety SOP"

        else:

            return "PDF Document"

    elif extension in [".png",".jpg",".jpeg"]:

        return "OCR Image"

    elif extension==".csv":

        return "Structured Dataset"

    elif extension==".txt":

        return "Text Document"

    return "Other"

inventory_df["Category"] = inventory_df.apply(

    classify_document,

    axis=1

)

In [32]:
# ==========================================================
# Metadata Preview
# ==========================================================

display(

    inventory_df[

        [

            "Document_ID",

            "File_Name",

            "Department",

            "Category",

            "File_Size_MB",

            "MIME_Type"

        ]

    ].head(20)

)

,Document_ID,File_Name,Department,Category,File_Size_MB,MIME_Type
0,DOC00001,86261.jpg,Operations,OCR Image,0.01,image/jpeg
1,DOC00002,86797.jpg,Operations,OCR Image,0.01,image/jpeg
2,DOC00003,86798.jpg,Operations,OCR Image,0.01,image/jpeg
3,DOC00004,86801.jpg,Operations,OCR Image,0.01,image/jpeg
4,DOC00005,86808.jpg,Operations,OCR Image,0.01,image/jpeg
5,DOC00006,86839.jpg,Operations,OCR Image,0.01,image/jpeg
6,DOC00007,86860.jpg,Operations,OCR Image,0.01,image/jpeg
7,DOC00008,86861.jpg,Operations,OCR Image,0.01,image/jpeg
8,DOC00009,86868.jpg,Operations,OCR Image,0.01,image/jpeg
9,DOC00010,86877.jpg,Operations,OCR Image,0.01,image/jpeg


In [33]:
# ==========================================================
# Metadata Validation
# ==========================================================

assert inventory_df["Department"].notna().all()

assert inventory_df["Category"].notna().all()

assert inventory_df["File_Size_MB"].notna().all()

assert inventory_df["Readable"].notna().all()

print("Metadata Validation Passed")

Metadata Validation Passed


In [34]:
# ==========================================================
# Metadata Statistics
# ==========================================================

print("="*60)

print("Metadata Statistics")

print("="*60)

print(

    inventory_df[

        [

            "Department",

            "Category"

        ]

    ].value_counts()

)

Metadata Statistics


Department   Category            
Operations   OCR Image               18
Engineering  PDF Document            18
Safety       Safety SOP              11
Quality      OCR Image                6
Maintenance  Maintenance Document     6
Name: count, dtype: int64


In [35]:
logging.info("Metadata Extraction Completed")

print()

print("="*60)

print("MILESTONE 3 COMPLETED")

print("="*60)


MILESTONE 3 COMPLETED


# Stage 3 — Enterprise File Fingerprinting

## Objective

Every enterprise document should have a unique cryptographic fingerprint.

This stage generates SHA-256 hashes for every document to:

- Detect duplicate files
- Verify file integrity
- Support incremental data ingestion
- Enable reproducible pipelines

Output

Document Hash Table

In [36]:
# ==========================================================
# SHA256 Generator
# ==========================================================

def generate_sha256(file_path: Path, chunk_size: int = 8192) -> str:
    """
    Generate SHA-256 hash for a file.

    Parameters
    ----------
    file_path : Path
        Path to the file.

    chunk_size : int
        Bytes read per iteration.

    Returns
    -------
    str
        SHA-256 hexadecimal digest.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [37]:
# ==========================================================
# Generate Hashes
# ==========================================================

print("Generating SHA-256 hashes...")

inventory_df["SHA256"] = inventory_df["Absolute_Path"].apply(
    lambda x: generate_sha256(Path(x))
)

print("Hash generation completed.")

Generating SHA-256 hashes...


Hash generation completed.


In [38]:
# ==========================================================
# Preview Hashes
# ==========================================================

display(

    inventory_df[

        [

            "Document_ID",

            "File_Name",

            "SHA256"

        ]

    ].head(10)

)

,Document_ID,File_Name,SHA256
0,DOC00001,86261.jpg,578d59e531b73c62d0f7a4ad3a28be79b7f996fcaaa077...
1,DOC00002,86797.jpg,a17eac2aeeceed94543ed66c3d56ff47214c493574c809...
2,DOC00003,86798.jpg,fa93c3655bba07c8570e493f6d9d6a2acd8df89fcf435d...
3,DOC00004,86801.jpg,4207043b7303c962d04a9dee84f73a8827df40254596c1...
4,DOC00005,86808.jpg,b3568921a6c57f8fb896b58f4ba08fbb7bb0b498b3683f...
5,DOC00006,86839.jpg,fe7694b1cd6c6cb2fe65e2ae5f497a892a01fe4e55753f...
6,DOC00007,86860.jpg,0f89d7bda7da7238504e55ecab08da22b5a26d98af071b...
7,DOC00008,86861.jpg,59b799cc944602cb2fb8be1c4a40c862bbce4bb7936251...
8,DOC00009,86868.jpg,f235b5706065ad7985bf2cfa718d78e0a1bee12b88a8bb...
9,DOC00010,86877.jpg,8921d5f1e4ea040c69ecc97b9e233b501e2770f95cf93c...


In [39]:
# ==========================================================
# Duplicate Detection
# ==========================================================

inventory_df["Duplicate"] = inventory_df.duplicated(
    subset=["SHA256"],
    keep=False
)

In [40]:
# ==========================================================
# Duplicate Summary
# ==========================================================

duplicate_count = inventory_df["Duplicate"].sum()

print("=" * 60)
print("Duplicate Detection Summary")
print("=" * 60)

print(f"Duplicate Files : {duplicate_count}")

Duplicate Detection Summary
Duplicate Files : 4


In [41]:
# ==========================================================
# Show Duplicates
# ==========================================================

duplicate_df = inventory_df[
    inventory_df["Duplicate"] == True
]

if duplicate_df.empty:

    print("No duplicate files detected.")

else:

    display(

        duplicate_df[

            [

                "Document_ID",

                "File_Name",

                "SHA256"

            ]

        ]

    )

,Document_ID,File_Name,SHA256
30,DOC00031,3GZC500930-178_en_B_Quick Start Guide of ABB L...,c37c363f5dee9faee093841806721e6f6243850b6f7814...
34,DOC00035,ENG-MAN-M504-ABB-001.pdf,c37c363f5dee9faee093841806721e6f6243850b6f7814...
44,DOC00045,P3U_en_M_J006_ANSI_web (1).pdf,41c96d8c910b7e30cc66ba3ccb481eb5c0e4bb213da4d7...
45,DOC00046,P3U_en_M_J006_ANSI_web.pdf,41c96d8c910b7e30cc66ba3ccb481eb5c0e4bb213da4d7...


In [42]:
# ==========================================================
# Hash Validation
# ==========================================================

assert inventory_df["SHA256"].notna().all()

assert inventory_df["SHA256"].str.len().eq(64).all()

print("SHA256 Validation Passed")

SHA256 Validation Passed


In [43]:
# ==========================================================
# Export Hash Table
# ==========================================================

hash_table = inventory_df[
    [

        "Document_ID",

        "File_Name",

        "SHA256"

    ]

]

hash_output = INVENTORY_DIR / "file_hashes.csv"

hash_table.to_csv(

    hash_output,

    index=False

)

print(f"Hash table saved to:\n{hash_output}")

Hash table saved to:
C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory\file_hashes.csv


In [44]:
# ==========================================================
# Stage Completion
# ==========================================================

logging.info("SHA256 Hash Generation Completed")

print()
print("=" * 60)
print("MILESTONE 4 COMPLETED")
print("=" * 60)


MILESTONE 4 COMPLETED


# Stage 4 — Enterprise Inventory Generation

## Objective

Transform all discovered document information into a unified enterprise inventory.

This inventory becomes the master catalog for every downstream notebook.

Outputs

- document_inventory.csv
- folder_summary.json
- dataset_statistics.json

In [45]:
# ==========================================================
# Final Inventory
# ==========================================================

inventory_df = inventory_df.sort_values(

    by="Document_ID"

).reset_index(drop=True)

print("Inventory Sorted")

Inventory Sorted


In [46]:
# ==========================================================
# Inventory Validation
# ==========================================================

required_columns = [

    "Document_ID",

    "File_Name",

    "Extension",

    "Folder",

    "Department",

    "Category",

    "SHA256"

]

missing = [

    col

    for col in required_columns

    if col not in inventory_df.columns

]

assert len(missing)==0, f"Missing Columns : {missing}"

print("Inventory Validation Passed")

Inventory Validation Passed


In [47]:
# ==========================================================
# Dataset Statistics
# ==========================================================

dataset_statistics = {

    "Total Documents":

        len(inventory_df),

    "PDF Files":

        int(

            (inventory_df["Extension"]==".pdf").sum()

        ),

    "Images":

        int(

            inventory_df["Extension"]

            .isin(

                [".png",".jpg",".jpeg"]

            ).sum()

        ),

    "CSV":

        int(

            (inventory_df["Extension"]==".csv").sum()

        ),

    "TXT":

        int(

            (inventory_df["Extension"]==".txt").sum()

        ),

    "Departments":

        int(

            inventory_df["Department"]

            .nunique()

        ),

    "Duplicate Files":

        int(

            inventory_df["Duplicate"].sum()

        ),

    "Total Size (MB)":

        round(

            inventory_df["File_Size_MB"].sum(),

            2

        ),

    "Average File Size (MB)":

        round(

            inventory_df["File_Size_MB"].mean(),

            2

        )

}

dataset_statistics

{'Total Documents': 59,
 'PDF Files': 35,
 'Images': 24,
 'CSV': 0,
 'TXT': 0,
 'Departments': 5,
 'Duplicate Files': 4,
 'Total Size (MB)': np.float64(417.31),
 'Average File Size (MB)': np.float64(7.07)}

In [48]:
# ==========================================================
# Folder Summary
# ==========================================================

folder_summary = (

    inventory_df

    .groupby(

        [

            "Department",

            "Folder"

        ]

    )

    .size()

    .reset_index(

        name="Document_Count"

    )

)

folder_summary

,Department,Folder,Document_Count
0,Engineering,abb,7
1,Engineering,atlas_copco,4
2,Engineering,schneider,5
3,Engineering,siemens,2
4,Maintenance,maintenance,6
5,Operations,equipment_labels,12
6,Operations,gauges,6
7,Quality,inspection_forms,6
8,Safety,safety_and_regulations,11


In [49]:
# ==========================================================
# Category Summary
# ==========================================================

category_summary = (

    inventory_df

    .groupby(

        "Category"

    )

    .size()

    .reset_index(

        name="Count"

    )

)

category_summary

,Category,Count
0,Maintenance Document,6
1,OCR Image,24
2,PDF Document,18
3,Safety SOP,11


In [50]:
# ==========================================================
# Largest Files
# ==========================================================

largest_files = (

    inventory_df

    .sort_values(

        "File_Size_MB",

        ascending=False

    )

    .head(10)

)

largest_files

,Document_ID,File_Name,Extension,Folder,Relative_Path,Absolute_Path,File_Size_Bytes,File_Size_MB,Last_Modified,MIME_Type,Readable,Department,Manufacturer,Category,SHA256,Duplicate
47,DOC00048,ENG-MAN-CNC602-SIEMENS-001.pdf,.pdf,siemens,data\raw\manuals\siemens\ENG-MAN-CNC602-SIEMEN...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,255563305,243.72,2026-07-11 14:23:53,application/pdf,True,Engineering,Siemens,PDF Document,20ec97056d7aa9aae56d3e2d6f1eaec6b3aa6bce407fd0...,False
43,DOC00044,M241-UserGuide-EN-EIO0000004267-05.pdf,.pdf,schneider,data\raw\manuals\schneider\M241-UserGuide-EN-E...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,28319147,27.01,2026-07-11 14:26:16,application/pdf,True,Engineering,Schneider Electric,PDF Document,ebe0f589511534d41211a6c02c922e008082aa37ecebf2...,False
46,DOC00047,ENG-MAN-CNC601-SIEMENS-001.pdf,.pdf,siemens,data\raw\manuals\siemens\ENG-MAN-CNC601-SIEMEN...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,26403614,25.18,2026-07-11 14:22:52,application/pdf,True,Engineering,Siemens,PDF Document,eca7156ed3429dbe3240987b7b3d5ff4ff55dfc3d43b71...,False
42,DOC00043,ENG-MAN-M502-SCHNEIDER-001.pdf,.pdf,schneider,data\raw\manuals\schneider\ENG-MAN-M502-SCHNEI...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,20410342,19.46,2026-07-11 14:27:44,application/pdf,True,Engineering,Schneider Electric,PDF Document,b612b0a75439afd655f33cc98c8cfc24de5f86d314b024...,False
27,DOC00028,MFR5408600 v03.00_Web.pdf,.pdf,maintenance,data\raw\maintenance\MFR5408600 v03.00_Web.pdf,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,15497689,14.78,2026-07-11 15:07:28,application/pdf,True,Maintenance,Nexus Industrial,Maintenance Document,293cbdbcd3afa00a9822a44c87f4c0c3b74b983835bac9...,False
25,DOC00026,GEX2563900EN_PIXStandard.pdf,.pdf,maintenance,data\raw\maintenance\GEX2563900EN_PIXStandard.pdf,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,13938716,13.29,2026-07-11 15:07:08,application/pdf,True,Maintenance,Nexus Industrial,Maintenance Document,546b101b2811883e5a7f9fd1ad8a880fac8adc3f001114...,False
41,DOC00042,ENG-MAN-M501-SCHNEIDER-001.pdf,.pdf,schneider,data\raw\manuals\schneider\ENG-MAN-M501-SCHNEI...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,9327318,8.90,2026-07-11 14:28:39,application/pdf,True,Engineering,Schneider Electric,PDF Document,6a4e9d36e91b78fa3e0ca4d599c30a6e1550176311bc3d...,False
45,DOC00046,P3U_en_M_J006_ANSI_web.pdf,.pdf,schneider,data\raw\manuals\schneider\P3U_en_M_J006_ANSI_...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,8554057,8.16,2026-07-11 14:27:08,application/pdf,True,Engineering,Schneider Electric,PDF Document,41c96d8c910b7e30cc66ba3ccb481eb5c0e4bb213da4d7...,True
44,DOC00045,P3U_en_M_J006_ANSI_web (1).pdf,.pdf,schneider,data\raw\manuals\schneider\P3U_en_M_J006_ANSI_...,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,8554057,8.16,2026-07-11 14:29:06,application/pdf,True,Engineering,Schneider Electric,PDF Document,41c96d8c910b7e30cc66ba3ccb481eb5c0e4bb213da4d7...,True
37,DOC00038,2935081743_L.pdf,.pdf,atlas_copco,data\raw\manuals\atlas_copco\2935081743_L.pdf,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6122246,5.84,2026-07-11 14:38:05,application/pdf,True,Engineering,Atlas Copco,PDF Document,ba1d8b88869553a1ff4dbb20005770adb788552555bb83...,False


In [51]:
# ==========================================================
# Statistics Report
# ==========================================================

print("="*70)

print("ENTERPRISE DATASET SUMMARY")

print("="*70)

for key,value in dataset_statistics.items():

    print(f"{key:<30}: {value}")

print("="*70)

ENTERPRISE DATASET SUMMARY
Total Documents               : 59
PDF Files                     : 35
Images                        : 24
CSV                           : 0
TXT                           : 0
Departments                   : 5
Duplicate Files               : 4
Total Size (MB)               : 417.31
Average File Size (MB)        : 7.07


In [52]:
# ==========================================================
# Export Inventory
# ==========================================================

inventory_output = (

    INVENTORY_DIR/

    "document_inventory.csv"

)

inventory_df.to_csv(

    inventory_output,

    index=False

)

print(inventory_output)

C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory\document_inventory.csv


In [53]:
# ==========================================================
# Export Folder Summary
# ==========================================================

folder_output = (

    INVENTORY_DIR/

    "folder_summary.json"

)

folder_summary.to_json(

    folder_output,

    orient="records",

    indent=4

)

print(folder_output)

C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory\folder_summary.json


In [54]:
# ==========================================================
# Export Statistics
# ==========================================================

statistics_output = (

    INVENTORY_DIR/

    "dataset_statistics.json"

)

with open(

    statistics_output,

    "w"

) as f:

    json.dump(

        dataset_statistics,

        f,

        indent=4

    )

print(statistics_output)

C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory\dataset_statistics.json


In [55]:
# ==========================================================
# Verify Exports
# ==========================================================

outputs = [

    inventory_output,

    hash_output,

    folder_output,

    statistics_output

]

for file in outputs:

    print(

        file.name,

        ":",

        file.exists()

    )

document_inventory.csv : True
file_hashes.csv : True
folder_summary.json : True
dataset_statistics.json : True


In [56]:
# ==========================================================
# Final Preview
# ==========================================================

display(

    inventory_df.head(20)

)

,Document_ID,File_Name,Extension,Folder,Relative_Path,Absolute_Path,File_Size_Bytes,File_Size_MB,Last_Modified,MIME_Type,Readable,Department,Manufacturer,Category,SHA256,Duplicate
0,DOC00001,86261.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86261.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,10120,0.01,2026-07-11 15:33:19,image/jpeg,True,Operations,Nexus Industrial,OCR Image,578d59e531b73c62d0f7a4ad3a28be79b7f996fcaaa077...,False
1,DOC00002,86797.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86797.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,7263,0.01,2026-07-11 15:32:46,image/jpeg,True,Operations,Nexus Industrial,OCR Image,a17eac2aeeceed94543ed66c3d56ff47214c493574c809...,False
2,DOC00003,86798.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86798.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,7689,0.01,2026-07-11 15:32:59,image/jpeg,True,Operations,Nexus Industrial,OCR Image,fa93c3655bba07c8570e493f6d9d6a2acd8df89fcf435d...,False
3,DOC00004,86801.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86801.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,5798,0.01,2026-07-11 15:33:38,image/jpeg,True,Operations,Nexus Industrial,OCR Image,4207043b7303c962d04a9dee84f73a8827df40254596c1...,False
4,DOC00005,86808.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86808.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6020,0.01,2026-07-11 15:33:31,image/jpeg,True,Operations,Nexus Industrial,OCR Image,b3568921a6c57f8fb896b58f4ba08fbb7bb0b498b3683f...,False
5,DOC00006,86839.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86839.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6696,0.01,2026-07-11 15:34:20,image/jpeg,True,Operations,Nexus Industrial,OCR Image,fe7694b1cd6c6cb2fe65e2ae5f497a892a01fe4e55753f...,False
6,DOC00007,86860.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86860.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6801,0.01,2026-07-11 15:32:52,image/jpeg,True,Operations,Nexus Industrial,OCR Image,0f89d7bda7da7238504e55ecab08da22b5a26d98af071b...,False
7,DOC00008,86861.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86861.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6290,0.01,2026-07-11 15:31:38,image/jpeg,True,Operations,Nexus Industrial,OCR Image,59b799cc944602cb2fb8be1c4a40c862bbce4bb7936251...,False
8,DOC00009,86868.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86868.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,6656,0.01,2026-07-11 15:31:14,image/jpeg,True,Operations,Nexus Industrial,OCR Image,f235b5706065ad7985bf2cfa718d78e0a1bee12b88a8bb...,False
9,DOC00010,86877.jpg,.jpg,equipment_labels,data\raw\images\equipment_labels\86877.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,7727,0.01,2026-07-11 15:33:55,image/jpeg,True,Operations,Nexus Industrial,OCR Image,8921d5f1e4ea040c69ecc97b9e233b501e2770f95cf93c...,False


In [57]:
# ==========================================================
# Final Assertions
# ==========================================================

assert inventory_output.exists()

assert hash_output.exists()

assert folder_output.exists()

assert statistics_output.exists()

print("All Outputs Generated Successfully")

All Outputs Generated Successfully


In [58]:
# ==========================================================
# Notebook Completion
# ==========================================================

logging.info("Notebook 01 Completed Successfully")

print()

print("="*80)

print("NOTEBOOK 01 COMPLETED")

print("="*80)


NOTEBOOK 01 COMPLETED


In [59]:
# ==========================================================
# Next Notebook
# ==========================================================

print("""

Notebook 02

Enterprise Document Processing Pipeline

Input

document_inventory.csv

Output

processed_documents.parquet

ocr_queue.csv

text_corpus.parquet

""")



Notebook 02

Enterprise Document Processing Pipeline

Input

document_inventory.csv

Output

processed_documents.parquet

ocr_queue.csv

text_corpus.parquet




In [60]:
# ==========================================================
# Verify Departments
# ==========================================================

print("=" * 60)
print("ENTERPRISE DEPARTMENTS")
print("=" * 60)

departments = sorted(inventory_df["Department"].unique())

print(f"Total Departments : {len(departments)}\n")

for i, dept in enumerate(departments, start=1):
    count = (inventory_df["Department"] == dept).sum()
    print(f"{i}. {dept:<15} -> {count} documents")

ENTERPRISE DEPARTMENTS
Total Departments : 5

1. Engineering     -> 18 documents
2. Maintenance     -> 6 documents
3. Operations      -> 18 documents
4. Quality         -> 6 documents
5. Safety          -> 11 documents


In [61]:
# ==========================================================
# One Example Document From Each Department
# ==========================================================

examples = (
    inventory_df
    .groupby("Department")
    .first()
    .reset_index()
)

display(
    examples[
        [
            "Department",
            "File_Name",
            "Category",
            "Folder",
            "Extension"
        ]
    ]
)

,Department,File_Name,Category,Folder,Extension
0,Engineering,3GZC500930-178_en_B_Quick Start Guide of ABB L...,PDF Document,abb,.pdf
1,Maintenance,990-5712_InRow RD DX Direct Expansion Air Cond...,Maintenance Document,maintenance,.pdf
2,Operations,86261.jpg,OCR Image,equipment_labels,.jpg
3,Quality,images (1).jpg,OCR Image,inspection_forms,.jpg
4,Safety,BBFACT01.pdf,Safety SOP,safety_and_regulations,.pdf


In [62]:
# ==========================================================
# Category Distribution
# ==========================================================

category_summary = (
    inventory_df
    .groupby("Category")
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(category_summary)

,Category,Count
1,OCR Image,24
2,PDF Document,18
3,Safety SOP,11
0,Maintenance Document,6


In [63]:
# ==========================================================
# Folder Verification
# ==========================================================

folder_summary = (
    inventory_df
    .groupby(["Department", "Folder"])
    .size()
    .reset_index(name="Documents")
)

display(folder_summary)

,Department,Folder,Documents
0,Engineering,abb,7
1,Engineering,atlas_copco,4
2,Engineering,schneider,5
3,Engineering,siemens,2
4,Maintenance,maintenance,6
5,Operations,equipment_labels,12
6,Operations,gauges,6
7,Quality,inspection_forms,6
8,Safety,safety_and_regulations,11


In [64]:
# ==========================================================
# Enterprise Inventory Overview
# ==========================================================

print("=" * 90)
print("ENTERPRISE INVENTORY")
print("=" * 90)

display(
    inventory_df[
        [
            "Document_ID",
            "Department",
            "Category",
            "File_Name",
            "Extension",
            "File_Size_MB",
            "Duplicate"
        ]
    ]
)

ENTERPRISE INVENTORY


,Document_ID,Department,Category,File_Name,Extension,File_Size_MB,Duplicate
0,DOC00001,Operations,OCR Image,86261.jpg,.jpg,0.01,False
1,DOC00002,Operations,OCR Image,86797.jpg,.jpg,0.01,False
2,DOC00003,Operations,OCR Image,86798.jpg,.jpg,0.01,False
3,DOC00004,Operations,OCR Image,86801.jpg,.jpg,0.01,False
4,DOC00005,Operations,OCR Image,86808.jpg,.jpg,0.01,False
5,DOC00006,Operations,OCR Image,86839.jpg,.jpg,0.01,False
6,DOC00007,Operations,OCR Image,86860.jpg,.jpg,0.01,False
7,DOC00008,Operations,OCR Image,86861.jpg,.jpg,0.01,False
8,DOC00009,Operations,OCR Image,86868.jpg,.jpg,0.01,False
9,DOC00010,Operations,OCR Image,86877.jpg,.jpg,0.01,False
